# Phase 1 — NHTSA Recall Data Ingestion and Cleaning

Ingest the NHTSA `FLAT_RCL_POST_2010` flat file, filter to vehicle
recalls, normalise manufacturer names, and tag each recall with an
OEM region/country and component group. Output feeds the NLP pipeline
(notebook 02) and the Power BI star schema.

**Scope**: US-market recalls, 2010 onwards.

In [ ]:
import pandas as pd
import zipfile
import numpy as np
import requests
from pathlib import Path

## Download and extract the raw NHTSA flat file

~15 MB ZIP. Saved under `data/raw/` (gitignored) so reruns don't
re-download.

In [ ]:
Path('../data/raw').mkdir(parents=True, exist_ok=True)

url = "https://static.nhtsa.gov/odi/ffdd/rcl/FLAT_RCL_POST_2010.zip"
save_path = Path('../data/raw/FLAT_RCL_POST_2010.zip')

response = requests.get(url)

with open(save_path, 'wb') as f:
    f.write(response.content)

print(f"Saved {len(response.content) / 1e6:.1f} MB to {save_path}")

In [ ]:
with zipfile.ZipFile(save_path, 'r') as z:
    z.extractall('../data/raw')

## Load the raw data

NHTSA's flat file deviates from its own data dictionary: tab-delimited
(not pipe), 29 columns (not 26), `latin-1` encoded, and a handful of
malformed rows. All four were discovered empirically — the documented
schema can't be trusted.

In [ ]:
# NHTSA flat file: tab-delimited, latin-1 encoded, 29 columns
# (3 more than the documented schema — see markdown above)

text_path = Path('../data/raw/FLAT_RCL_POST_2010.txt')
df_raw = pd.read_csv(
    text_path,
    sep='\t',
    header=None,
    encoding='latin-1',
    low_memory=False,
    on_bad_lines='skip'
)
print(df_raw.shape)


In [ ]:
columns = [
    'RECORD_ID', 'CAMPNO', 'MAKETXT', 'MODELTXT', 'YEARTXT', 'MFGCAMPNO',
    'COMPNAME', 'MFGNAME', 'BGMAN', 'ENDMAN', 'RCLTYPECD',
    'POTAFF', 'ODATE', 'INFLUENCED_BY', 'MFGTXT', 'RCDATE',
    'DATEA', 'RPNO', 'FMVSS', 'DESC_DEFECT', 'CONEQUENCE_DEFECT',
    'CORRECTIVE_ACTION', 'NOTES', 'RCL_CMPT_ID', 'MFR_COMP_NAME',
    'MFR_COMP_DESC', 'MFR_COMP_PTNO', 'POTENTIALLY_AFFECTED_FLAG',
    'COMPLETION_RATE'
]

In [ ]:
df = df_raw.copy()
df.columns =columns
df. head(3)

## Filter to vehicle recalls only

NHTSA mixes vehicle (`V`), equipment (`E`), tyre (`T`), and child-seat
(`C`) recalls. Only `V` is in scope. `RCLTYPECD` values have trailing
whitespace and need stripping before the filter.

In [ ]:
# Strip whitespace before filtering — RCLTYPECD values in raw file
# have trailing spaces

df_vehicle = df[df['RCLTYPECD'] == 'V'].copy()
print(df_vehicle.shape)
print(df_vehicle['RCDATE'].dtype)
print(df_vehicle['RCDATE'].min(), df_vehicle['RCDATE'].max())

In [ ]:
# Convert 'RCDATE' from integer to proper datetime

df_vehicle['RCDATE'] = pd.to_datetime(df_vehicle['RCDATE'].astype(str), format='%Y%m%d', errors='coerce')
print(df_vehicle['RCDATE'].dtype)
print(df_vehicle['RCDATE'].min(), df_vehicle['RCDATE'].max())
df_vehicle['RCDATE'].head(10)

In [ ]:
# Create new column 'RECALL_YEAR'.
# This will serve as the primary time axis in Power BI

df_vehicle['RECALL_YEAR'] = df_vehicle['RCDATE'].dt.year

In [ ]:
print(df_vehicle['MFGNAME'].nunique())
print(df_vehicle['MFGNAME'].value_counts())

## Manufacturer name normalisation

1,267 unique `MFGNAME` strings for ~100 real OEMs — Mercedes-Benz
alone appears as five variants. Without normalisation, any per-OEM
aggregation is wrong.

**Approach**: keyword map for the top ~25 OEMs (deterministic,
handles cases like Volvo Cars vs Volvo Trucks that fuzzy matching
would wrongly merge), then a `rapidfuzz` pass at threshold 92 on the
remainder.

In [ ]:
mb_variants = df_vehicle[df_vehicle['MFGNAME'].str.contains('Mercedes', case=False, na=False)]['MFGNAME'].value_counts()
mb_variants

In [ ]:
# Each keyword maps to a canonical name. A single keyword like
# 'mercedes' catches every variant (LLC, LLC., DBA SPRINTER, etc.)
keyword_map = {
    'mercedes':       'Mercedes-Benz',
    'daimler truck':  'Daimler Trucks',
    'daimler van':    'Mercedes-Benz',     # Sprinter is sold as MB in the US   
    'bmw':            'BMW',
    'toyota':         'Toyota',
    'ford':           'Ford',
    'honda':          'Honda',
    'volkswagen':     'Volkswagen',
    'porsche':        'Porsche',
    'audi':           'Audi',
    'jaguar':         'Jaguar Land Rover',
    'land rover':     'Jaguar Land Rover',
    'volvo car':      'Volvo Cars',
    'volvo truck':    'Volvo Trucks',
    'nissan':         'Nissan',
    'general motors': 'General Motors',
    'chrysler':       'Stellantis (Chrysler/FCA)',
    'fca us':         'Stellantis (Chrysler/FCA)',
    'hyundai':        'Hyundai',
    'kia':            'Kia',
    'subaru':         'Subaru',
    'mitsubishi':     'Mitsubishi',
    'tesla':          'Tesla',
    'ferrari':        'Ferrari',
    'harley':         'Harley-Davidson',
    'paccar':         'PACCAR',
    'navistar':       'Navistar',
    'mack truck':     'Mack Trucks',
    'maserati':       'Maserati',
    'bentley':        'Bentley',
    'lamborghini':    'Lamborghini',
    'mazda':          'Mazda',
    'hino':           'Hino',
    'international motors': 'Navistar',
}

In [ ]:
def clean_name(name):
    name = name.lower()
    name = name.replace(',', '').replace('.', '')

    suffixes = {'inc', 'llc', 'corp', 'corporation', 'co', 'ltd', 'lp', 'company'}
    words = [w for w in name.split() if w not in suffixes]

    return ' '.join(words).strip().title()

In [ ]:
# Canonical name function

def get_canonical_name(name):
    if pd.isna(name):
        return 'Unknown'

    name_lower = name.lower()

    for keyword, canonical in keyword_map.items():
        if keyword in name_lower:
            return canonical 
    
    return clean_name(name)

In [ ]:
df_vehicle['MFG_CANONICAL'] = df_vehicle['MFGNAME'].apply(get_canonical_name)

print("Unique names before: ", df_vehicle['MFGNAME'].nunique())
print("Unique names after: ", df_vehicle['MFG_CANONICAL'].nunique())

## OEM region and country tagging

Each canonical manufacturer is tagged with HQ region and country.
Drives the European-OEM filter in Power BI and the US-vs-EU contrast
in the final write-up. The long tail of RV, bus, and specialty
makers is bucketed as `Other`.

In [ ]:
oem_info = {
    # ---- Europe: Germany ----
    'Mercedes-Benz':              ('Europe', 'Germany'),
    'BMW':                        ('Europe', 'Germany'),
    'Volkswagen':                 ('Europe', 'Germany'),
    'Audi':                       ('Europe', 'Germany'),
    'Porsche':                    ('Europe', 'Germany'),

    # ---- Europe: UK ----
    'Jaguar Land Rover':          ('Europe', 'UK'),
    'Bentley':                    ('Europe', 'UK'),

    # ---- Europe: Sweden ----
    'Volvo Cars':                 ('Europe', 'Sweden'),
    'Volvo Trucks':               ('Europe', 'Sweden'),

    # ---- Europe: Italy ----
    'Ferrari':                    ('Europe', 'Italy'),
    'Maserati':                   ('Europe', 'Italy'),
    'Lamborghini':                ('Europe', 'Italy'),

    # ---- Europe: multinational ----
    'Stellantis (Chrysler/FCA)':  ('Europe', 'Multinational'),  # see note below

    # ---- North America: USA ----
    'Ford':                       ('North America', 'USA'),
    'General Motors':             ('North America', 'USA'),
    'Tesla':                      ('North America', 'USA'),
    'PACCAR':                     ('North America', 'USA'),
    'Navistar':                   ('North America', 'USA'),
    'Mack Trucks':                ('North America', 'USA'),
    'Harley-Davidson':            ('North America', 'USA'),

    # ---- Asia: Japan ----
    'Toyota':                     ('Asia', 'Japan'),
    'Honda':                      ('Asia', 'Japan'),
    'Nissan':                     ('Asia', 'Japan'),
    'Subaru':                     ('Asia', 'Japan'),
    'Mitsubishi':                 ('Asia', 'Japan'),
    'Mazda':                      ('Asia', 'Japan'),
    'Hino':                       ('Asia', 'Japan'),

    # ---- Asia: South Korea ----
    'Hyundai':                    ('Asia', 'South Korea'),
    'Kia':                        ('Asia', 'South Korea'),

    # ---- Special cases ----
    'Daimler Trucks':             ('Europe', 'Germany'), 
}



In [ ]:
df_vehicle['OEM_REGION'] = df_vehicle['MFG_CANONICAL'].map(
    lambda x: oem_info.get(x, ('Other', 'Other'))[0]
)

df_vehicle['OEM_COUNTRY'] = df_vehicle['MFG_CANONICAL'].map(
    lambda x: oem_info.get(x, ('Other', 'Other'))[1]
)

In [ ]:
df_vehicle['OEM_COUNTRY'].value_counts().head(20)

In [ ]:
df_vehicle[df_vehicle['OEM_REGION'] == 'Other']['MFG_CANONICAL'].value_counts().head(30)

In [ ]:
df_vehicle['COMPNAME'].value_counts().head(40)

In [ ]:
df_vehicle['COMPNAME'].str.split(':').str[0].value_counts()

## Component grouping

NHTSA's `COMPNAME` is a colon-delimited hierarchy. Splitting on the
first colon yields ~40 top-level categories; `component_map` collapses
these to 11 analytical groups.

This rule-based taxonomy is deliberately complementary to the NLP
topic model in notebook 02 — the interesting signal will be where
the two disagree. `Communication/Admin` and `Other` are kept in the
dataset but excluded from defect analysis.

In [ ]:
# Rule-based taxonomy 
# ~40 top-level COMPNAME categories collapsed to 11 groups.

component_map = {
    # ---- Software/Electronics ----
    # (Electrical is split: pure electrical stays here as "Electrical".
    # Software-specific recalls will be re-tagged from NLP later.)

    # ---- Electrical ----
    'ELECTRICAL SYSTEM':                  'Electrical',

    # ---- Safety Restraint ----
    'AIR BAGS':                           'Safety Restraint',
    'SEAT BELTS':                         'Safety Restraint',
    'CHILD SEAT':                         'Safety Restraint',

    # ---- Braking ----
    'SERVICE BRAKES, HYDRAULIC':          'Braking',
    'SERVICE BRAKES, AIR':                'Braking',
    'SERVICE BRAKES, ELECTRIC':           'Braking',
    'SERVICE BRAKES':                     'Braking',
    'PARKING BRAKE':                      'Braking',

    # ---- Powertrain ----
    'POWER TRAIN':                        'Powertrain',
    'ENGINE AND ENGINE COOLING':          'Powertrain',
    'ENGINE':                             'Powertrain',
    'HYBRID PROPULSION SYSTEM':           'Powertrain',
    'FUEL/PROPULSION SYSTEM':             'Powertrain',
    'VEHICLE SPEED CONTROL':              'Powertrain',

    # ---- Fuel & Fire Risk ----
    'FUEL SYSTEM, GASOLINE':              'Fuel & Fire Risk',
    'FUEL SYSTEM, DIESEL':                'Fuel & Fire Risk',
    'FUEL SYSTEM, OTHER':                 'Fuel & Fire Risk',

    # ---- Steering & Suspension ----
    'STEERING':                           'Steering & Suspension',
    'SUSPENSION':                         'Steering & Suspension',

    # ---- Driver Assistance / ADAS ----
    'BACK OVER PREVENTION':               'Driver Assistance',
    'FORWARD COLLISION AVOIDANCE':        'Driver Assistance',
    'ELECTRONIC STABILITY CONTROL (ESC)': 'Driver Assistance',
    'LANE DEPARTURE':                     'Driver Assistance',
    'TRACTION CONTROL SYSTEM':            'Driver Assistance',

    # ---- Visibility & Lighting ----
    'EXTERIOR LIGHTING':                  'Visibility & Lighting',
    'INTERIOR LIGHTING':                  'Visibility & Lighting',
    'VISIBILITY':                         'Visibility & Lighting',
    'VISIBILITY/WIPER':                   'Visibility & Lighting',

    # ---- Structural & Body ----
    'STRUCTURE':                          'Structural & Body',
    'LATCHES/LOCKS/LINKAGES':             'Structural & Body',
    'SEATS':                              'Structural & Body',
    'TIRES':                              'Structural & Body',
    'WHEELS':                             'Structural & Body',
    'TRAILER HITCHES':                    'Structural & Body',

    # ---- Communication / Admin (excluded from defect analysis) ----
    'COMMUNICATION':                      'Communication/Admin',

    # ---- Other ----
    'EQUIPMENT':                          'Other',
    'EQUIPMENT ADAPTIVE/MOBILITY':        'Other',
    'UNKNOWN OR OTHER':                   'Other',
    'OTHER':                              'Other',
}

In [ ]:
df_vehicle['COMPONENT_GROUP'] = (
    df_vehicle['COMPNAME']
      .str.split(':').str[0]
      .str.strip()
      .map(component_map)
      .fillna('Other')
)